In [ ]:
import sys
sys.path.append('/home/satanka/local/nv5/idl92/lib/bridges')

# データ保存

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# テスト

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr
from idlpy import *

pt.del_data('*')

In [ ]:
import tempfile
import time
from pathlib import Path
from idlpy import IDL

def run_idl_batch_simple(commands, timeout=300, poll=0.5):
    """
    Run IDL commands through a temporary .pro batch file.
    Completion is judged only by a DONE sentinel file.
    """
    tmpdir = Path(tempfile.mkdtemp(prefix="idl_batch_"))
    tmpdir_idl = tmpdir.as_posix()

    done_file = tmpdir / "DONE"
    batch_file = tmpdir / "batch.pro"

    lines = list(commands)
    lines.append(f"openw, lun_done, '{done_file.as_posix()}', /get_lun")
    lines.append("printf, lun_done, 'DONE'")
    lines.append("free_lun, lun_done")

    batch_file.write_text("\n".join(lines) + "\n")

    t0 = time.time()
    IDL.run(f"@{batch_file.as_posix()}")

    while not done_file.exists():
        if time.time() - t0 > timeout:
            raise TimeoutError(
                f"IDL batch did not finish within {timeout} s. "
                f"Batch dir: {tmpdir_idl}"
            )
        time.sleep(poll)

    return tmpdir

In [ ]:
IDL.run("trange = ['2022-09-01/22:25:00', '2022-09-01/23:15:00']")
trange = IDL.trange
print(trange)

In [ ]:
tmpdir = run_idl_batch_simple([
    f"trange = {trange}",
    "thm_load_state, probe='a', trange=trange, /get_support_data",
    "thm_load_fgm, trange=trange, probe='a', datatype='all', /use_eclipse",
    "thm_load_mom, trange=trange, probe='a'",
    "thm_part_load, probe='a', datatype='peif', trange=trange, /forceload",
    "thm_part_load, probe='a', datatype='peef', trange=trange, /forceload",
    "thm_part_products, probe='a', datatype='peif', trange=trange, output='pa', units='flux', energy=[40,1000], mag_name='tha_fgl'",
    "thm_part_products, probe='a', datatype='peef', trange=trange, output='pa', units='flux', energy=[40,5000], mag_name='tha_fgl'",
    "get_data, 'tha_peif_flux_pa', data=d_peif",
    "get_data, 'tha_peef_flux_pa', data=d_peef",
    "peif_time = d_peif.x",
    "peif_flux = d_peif.y",
    "peif_pa = d_peif.v",
    "peef_time = d_peef.x",
    "peef_flux = d_peef.y",
    "peef_pa = d_peef.v",
], timeout=300)

print(tmpdir)

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

peif_time = np.asarray(IDL.peif_time)
peif_flux = np.asarray(IDL.peif_flux)
peif_pa   = np.asarray(IDL.peif_pa)

peef_time = np.asarray(IDL.peef_time)
peef_flux = np.asarray(IDL.peef_flux)
peef_pa   = np.asarray(IDL.peef_pa)

print("peif:", peif_time.shape, peif_flux.shape, peif_pa.shape)
print("peef:", peef_time.shape, peef_flux.shape, peef_pa.shape)

In [ ]:
def make_pa_flux_dataset(
    time_unix,
    flux,
    pa,
    *,
    varname="diff_number_flux",
    species="ion",
    datatype="peif",
    energy_range_ev=(50.0, 1000.0),
):
    """
    IDL/SPEDAS tplotの pitch-angle flux product を xarray.Dataset に変換する。

    time_unix: d.x, UNIX秒
    flux: d.y
    pa: d.v
    """

    time_unix = np.asarray(time_unix, dtype=np.float64)
    flux = np.asarray(flux)
    pa = np.asarray(pa)

    # time coordinate
    time = pd.to_datetime(time_unix, unit="s", utc=True).tz_convert(None)

    # pitch-angle axis
    if pa.ndim == 1:
        pa_axis = pa.astype(np.float64)
    elif pa.ndim == 2:
        # まれにtime-dependent axisとして来る場合の保険
        if pa.shape[0] == len(time_unix):
            pa_axis = np.nanmedian(pa, axis=0)
        elif pa.shape[1] == len(time_unix):
            pa_axis = np.nanmedian(pa, axis=1)
        else:
            raise ValueError(f"Cannot identify PA axis from pa.shape={pa.shape}")
        pa_axis = pa_axis.astype(np.float64)
    else:
        raise ValueError(f"Unexpected pa.ndim={pa.ndim}, pa.shape={pa.shape}")

    ntime = len(time_unix)
    npa = len(pa_axis)

    # fluxを(time, pitch_angle)にそろえる
    if flux.shape == (ntime, npa):
        flux_t_pa = flux
    elif flux.shape == (npa, ntime):
        flux_t_pa = flux.T
    else:
        raise ValueError(
            f"Unexpected flux shape: {flux.shape}; "
            f"expected {(ntime, npa)} or {(npa, ntime)}"
        )

    flux_t_pa = np.asarray(flux_t_pa, dtype=np.float32)

    ds = xr.Dataset(
        data_vars={
            varname: (
                ("time", "pitch_angle"),
                flux_t_pa,
                {
                    "long_name": "Differential number flux",
                    "units": "cm^-2 s^-1 sr^-1 eV^-1",
                    "species": species,
                    "datatype": datatype,
                    "energy_min_eV": float(energy_range_ev[0]),
                    "energy_max_eV": float(energy_range_ev[1]),
                },
            )
        },
        coords={
            "time": ("time", time.values),
            "pitch_angle": (
                "pitch_angle",
                pa_axis,
                {
                    "long_name": "Pitch angle",
                    "units": "deg",
                },
            ),
        },
        attrs={
            "source": "THEMIS IDL/SPEDAS thm_part_products",
            "product": "pitch-angle differential number flux",
            "units_requested_in_thm_part_products": "flux",
            "energy_range_eV": f"{energy_range_ev[0]}-{energy_range_ev[1]}",
        },
    )

    return ds

In [ ]:
ds_peif = make_pa_flux_dataset(
    peif_time,
    peif_flux,
    peif_pa,
    species="ion",
    datatype="peif",
    energy_range_ev=(40.0, 1000.0),
)

ds_peef = make_pa_flux_dataset(
    peef_time,
    peef_flux,
    peef_pa,
    species="electron",
    datatype="peef",
    energy_range_ev=(40.0, 5000.0),
)

print(ds_peif)
print(ds_peef)

In [ ]:
out_dir = Path("/mnt/j/observation_data/themis/tha/idl_output")
out_dir.mkdir(parents=True, exist_ok=True)

peif_nc = out_dir / "tha_peif_flux_pa_40_1000eV_20220901_2225_2315.nc"
peef_nc = out_dir / "tha_peef_flux_pa_40_5000eV_20220901_2225_2315.nc"

ds_peif.to_netcdf(peif_nc)
ds_peef.to_netcdf(peef_nc)

print(peif_nc)
print(peef_nc)

In [ ]:
ds_peif_check = xr.open_dataset(peif_nc)
ds_peef_check = xr.open_dataset(peef_nc)

print(ds_peif_check)
print(ds_peef_check)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

da = ds_peef_check["diff_number_flux"]

plt.figure(figsize=(10, 4))
plt.pcolormesh(
    da["time"].values,
    da["pitch_angle"].values,
    da.values.T,
    shading="auto",
    norm=LogNorm(),
    cmap='turbo'
)
plt.colorbar(label=r"Diff. number flux [cm$^{-2}$ s$^{-1}$ sr$^{-1}$ eV$^{-1}$]")
plt.ylabel("Pitch angle [deg]")
plt.xlabel("Time [UT]")
plt.ylim(0, 180)
plt.tight_layout()
plt.show()

# omniflux

In [ ]:
tmpdir = run_idl_batch_simple([
    f"trange = {trange}",
    "thm_load_state, probe='a', trange=trange, /get_support_data, /no_update",
    "thm_load_fgm, trange=trange, probe='a', datatype='all', /use_eclipse, /no_update",
    "thm_load_mom, trange=trange, probe='a', /no_update",

    "thm_part_load, probe='a', datatype='peir', trange=trange, /forceload, /no_update",
    "thm_part_load, probe='a', datatype='peer', trange=trange, /forceload, /no_update",

    "thm_part_products, probe='a', datatype='peir', trange=trange, outputs='energy', units='flux'",
    "thm_part_products, probe='a', datatype='peer', trange=trange, outputs='energy', units='flux'",

    "get_data, 'tha_peir_flux_energy', data=d_peir_omni",
    "get_data, 'tha_peer_flux_energy', data=d_peer_omni",

    "peir_omni_time = d_peir_omni.x",
    "peir_omni_flux = d_peir_omni.y",
    "peir_omni_energy = d_peir_omni.v",

    "peer_omni_time = d_peer_omni.x",
    "peer_omni_flux = d_peer_omni.y",
    "peer_omni_energy = d_peer_omni.v",
], timeout=300)

In [ ]:
import numpy as np

peir_omni_time = np.asarray(IDL.peir_omni_time)
peir_omni_flux = np.asarray(IDL.peir_omni_flux)
peir_omni_energy = np.asarray(IDL.peir_omni_energy)

peer_omni_time = np.asarray(IDL.peer_omni_time)
peer_omni_flux = np.asarray(IDL.peer_omni_flux)
peer_omni_energy = np.asarray(IDL.peer_omni_energy)

print("peir:", peir_omni_time.shape, peir_omni_flux.shape, peir_omni_energy.shape)
print("peer:", peer_omni_time.shape, peer_omni_flux.shape, peer_omni_energy.shape)

In [ ]:
import pandas as pd
import xarray as xr
from pathlib import Path

def make_omni_flux_dataset(
    time_unix,
    flux,
    energy,
    *,
    datatype,
    probe="a",
):
    time_unix = np.asarray(time_unix, dtype=np.float64)
    flux = np.asarray(flux)
    energy = np.asarray(energy)

    time = pd.to_datetime(time_unix, unit="s", utc=True).tz_convert(None)

    # energy axis
    if energy.ndim == 1:
        energy_axis = energy.astype(float)
    elif energy.ndim == 2:
        if energy.shape[0] == len(time_unix):
            energy_axis = np.nanmedian(energy, axis=0)
        elif energy.shape[1] == len(time_unix):
            energy_axis = np.nanmedian(energy, axis=1)
        else:
            raise ValueError(f"Cannot identify energy axis: {energy.shape}")
    else:
        raise ValueError(f"Unexpected energy shape: {energy.shape}")

    ntime = len(time_unix)
    nenergy = len(energy_axis)

    # fluxを(time, energy)にそろえる
    if flux.shape == (ntime, nenergy):
        flux_t_e = flux
    elif flux.shape == (nenergy, ntime):
        flux_t_e = flux.T
    else:
        raise ValueError(
            f"Unexpected flux shape: {flux.shape}; "
            f"expected {(ntime, nenergy)} or {(nenergy, ntime)}"
        )

    species = {"peir": "ion", "peer": "electron"}.get(datatype, "unknown")

    ds = xr.Dataset(
        data_vars={
            "diff_number_flux": (
                ("time", "energy"),
                np.asarray(flux_t_e, dtype=np.float32),
                {
                    "long_name": "Omnidirectional differential number flux",
                    "units": "cm^-2 s^-1 sr^-1 eV^-1",
                    "datatype": datatype,
                    "species": species,
                },
            )
        },
        coords={
            "time": ("time", time.values),
            "energy": (
                "energy",
                energy_axis.astype(float),
                {
                    "long_name": "Energy",
                    "units": "eV",
                },
            ),
        },
        attrs={
            "source": "THEMIS IDL/SPEDAS thm_part_products",
            "product": "omnidirectional energy spectrogram",
            "outputs": "energy",
            "units_requested_in_thm_part_products": "flux",
            "datatype": datatype,
            "species": species,
            "probe": probe,
        },
    )

    return ds

In [ ]:
out_dir_omni = Path("/mnt/j/observation_data/themis/tha/idl_output/omni_flux")
out_dir_omni.mkdir(parents=True, exist_ok=True)

ds_peir_omni = make_omni_flux_dataset(
    peir_omni_time,
    peir_omni_flux,
    peir_omni_energy,
    datatype="peir",
    probe="a",
)

ds_peer_omni = make_omni_flux_dataset(
    peer_omni_time,
    peer_omni_flux,
    peer_omni_energy,
    datatype="peer",
    probe="a",
)

peir_nc = out_dir_omni / "tha_peir_omni_diff_number_flux.nc"
peer_nc = out_dir_omni / "tha_peer_omni_diff_number_flux.nc"

ds_peir_omni.to_netcdf(peir_nc)
ds_peer_omni.to_netcdf(peer_nc)

print(peir_nc)
print(peer_nc)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

da = ds_peer_omni["diff_number_flux"]

arr = da.values.astype(float)
arr[arr <= 0] = np.nan
vmin, vmax = np.nanpercentile(arr[np.isfinite(arr)], [50, 99])

plt.figure(figsize=(10, 4))
plt.pcolormesh(
    da["time"].values,
    da["energy"].values,
    arr.T,
    shading="auto",
    norm=mcolors.LogNorm(vmin=vmin, vmax=vmax),
    cmap='turbo'
)
plt.yscale("log")
plt.colorbar(label=r"Diff. number flux [cm$^{-2}$ s$^{-1}$ sr$^{-1}$ eV$^{-1}$]")
plt.ylabel("Energy [eV]")
plt.xlabel("Time [UT]")
plt.tight_layout()
plt.show()

# エネルギーごとに取得

In [ ]:
import pyspedas as psp
import ergpyspedas.erg as ergpy
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr
from pathlib import Path
from idlpy import *

pt.del_data('*')

In [ ]:
import tempfile
import time
from pathlib import Path
from idlpy import IDL

def run_idl_batch_simple(commands, timeout=300, poll=0.5):
    """
    Run IDL commands through a temporary .pro batch file.
    Completion is judged only by a DONE sentinel file.
    """
    tmpdir = Path(tempfile.mkdtemp(prefix="idl_batch_"))
    tmpdir_idl = tmpdir.as_posix()

    done_file = tmpdir / "DONE"
    batch_file = tmpdir / "batch.pro"

    lines = list(commands)
    lines.append(f"openw, lun_done, '{done_file.as_posix()}', /get_lun")
    lines.append("printf, lun_done, 'DONE'")
    lines.append("free_lun, lun_done")

    batch_file.write_text("\n".join(lines) + "\n")

    t0 = time.time()
    IDL.run(f"@{batch_file.as_posix()}")

    while not done_file.exists():
        if time.time() - t0 > timeout:
            raise TimeoutError(
                f"IDL batch did not finish within {timeout} s. "
                f"Batch dir: {tmpdir_idl}"
            )
        time.sleep(poll)

    return tmpdir

In [ ]:
tmpdir = run_idl_batch_simple([
    f"trange = {trange}",
    "thm_part_load, probe='a', datatype='peif', trange=trange, /forceload",
    "thm_part_load, probe='a', datatype='peef', trange=trange, /forceload",

    "thm_part_products, probe='a', datatype='peif', trange=trange, outputs='energy', units='flux'",
    "thm_part_products, probe='a', datatype='peef', trange=trange, outputs='energy', units='flux'",

    "get_data, 'tha_peif_flux_energy', data=d_peif_energy",
    "get_data, 'tha_peef_flux_energy', data=d_peef_energy",

    "peif_energy_time = d_peif_energy.x",
    "peif_energy_flux = d_peif_energy.y",
    "peif_energy_axis = d_peif_energy.v",

    "peef_energy_time = d_peef_energy.x",
    "peef_energy_flux = d_peef_energy.y",
    "peef_energy_axis = d_peef_energy.v",
], timeout=300)

In [ ]:
import numpy as np

peif_energy_time = np.asarray(IDL.peif_energy_time)
peif_energy_flux = np.asarray(IDL.peif_energy_flux)
peif_energy_axis = np.asarray(IDL.peif_energy_axis)

peef_energy_time = np.asarray(IDL.peef_energy_time)
peef_energy_flux = np.asarray(IDL.peef_energy_flux)
peef_energy_axis = np.asarray(IDL.peef_energy_axis)

print("peif axis:", peif_energy_axis.shape)
print("peef axis:", peef_energy_axis.shape)

print(peif_energy_axis)
print(peef_energy_axis)

In [ ]:
def representative_energy_axis(E):
    E = np.asarray(E, dtype=float)

    if E.ndim == 1:
        axis = E
    elif E.ndim == 2:
        # どちらの軸がtimeかは状況依存なので、NaNを無視して中央値を取る
        # ここでは短い方をenergy軸とみなす
        if E.shape[0] <= E.shape[1]:
            axis = np.nanmedian(E, axis=1)
        else:
            axis = np.nanmedian(E, axis=0)
    else:
        raise ValueError(f"Unexpected energy axis shape: {E.shape}")

    axis = axis[np.isfinite(axis)]
    axis = np.unique(axis)

    # 昇順にそろえる
    axis = np.sort(axis)

    return axis

peif_E = representative_energy_axis(peif_energy_axis)
peef_E = representative_energy_axis(peef_energy_axis)

print("Ion ESA representative energies [eV]")
print(peif_E)

print("Electron ESA representative energies [eV]")
print(peef_E)

In [ ]:
def energy_centers_to_edges_log(E):
    E = np.asarray(E, dtype=float)
    E = E[np.isfinite(E) & (E > 0)]
    E = np.sort(E)

    if len(E) < 2:
        raise ValueError("Need at least two energy centers")

    mid = np.sqrt(E[:-1] * E[1:])

    first = E[0]**2 / mid[0]
    last = E[-1]**2 / mid[-1]

    edges = np.r_[first, mid, last]
    return edges

peif_edges = energy_centers_to_edges_log(peif_E)
peef_edges = energy_centers_to_edges_log(peef_E)

print("Ion bin edges [eV]")
print(peif_edges)

print("Electron bin edges [eV]")
print(peef_edges)

In [ ]:
def prepare_themis_esa_for_pa(
    trange,
    datatype="peif",
    probe="a",
    timeout=300,
    load_mom=True,
):
    """
    thm_part_products(outputs='pa') に必要なTHEMISデータをIDL/SPEDAS側に読む。
    datatype: 'peif' or 'peef'
    """

    if datatype not in ["peif", "peef"]:
        raise ValueError("datatype must be 'peif' or 'peef'")

    commands = [
        f"trange = {trange}",
        f"thm_load_state, probe='{probe}', trange=trange, /get_support_data, /no_update",
        f"thm_load_fgm, trange=trange, probe='{probe}', datatype='all', /use_eclipse, /no_update",
    ]

    if load_mom:
        commands.append(
            f"thm_load_mom, trange=trange, probe='{probe}', /no_update"
        )

    commands.append(
        f"thm_part_load, probe='{probe}', datatype='{datatype}', trange=trange, /forceload, /no_update"
    )

    tmpdir = run_idl_batch_simple(commands, timeout=timeout)
    return tmpdir

In [ ]:
def pa_flux_to_dataset(
    time_unix,
    flux,
    pa,
    *,
    datatype,
    probe="a",
    energy_range_ev=None,
):
    """
    IDL/SPEDAS thm_part_products(outputs='pa', units='flux') の結果を
    xarray.Dataset に変換する。
    """

    time_unix = np.asarray(time_unix, dtype=np.float64)
    flux = np.asarray(flux)
    pa = np.asarray(pa)

    time = pd.to_datetime(time_unix, unit="s", utc=True).tz_convert(None)

    # pitch angle axis
    if pa.ndim == 1:
        pa_axis = pa.astype(np.float64)
    elif pa.ndim == 2:
        # time-dependent axisとして返る場合の保険
        if pa.shape[0] == len(time_unix):
            pa_axis = np.nanmedian(pa, axis=0)
        elif pa.shape[1] == len(time_unix):
            pa_axis = np.nanmedian(pa, axis=1)
        else:
            raise ValueError(f"Cannot identify PA axis: pa.shape={pa.shape}")
        pa_axis = pa_axis.astype(np.float64)
    else:
        raise ValueError(f"Unexpected pa.shape={pa.shape}")

    ntime = len(time_unix)
    npa = len(pa_axis)

    # fluxを(time, pitch_angle)にそろえる
    if flux.shape == (ntime, npa):
        flux_t_pa = flux
    elif flux.shape == (npa, ntime):
        flux_t_pa = flux.T
    else:
        raise ValueError(
            f"Unexpected flux shape: {flux.shape}; "
            f"expected {(ntime, npa)} or {(npa, ntime)}"
        )

    flux_t_pa = np.asarray(flux_t_pa, dtype=np.float32)

    species = {
        "peif": "ion",
        "peef": "electron",
    }.get(datatype, "unknown")

    attrs = {
        "source": "THEMIS IDL/SPEDAS thm_part_products",
        "product": "pitch-angle differential number flux",
        "datatype": datatype,
        "species": species,
        "probe": probe,
        "units_requested_in_thm_part_products": "flux",
    }

    var_attrs = {
        "long_name": "Differential number flux",
        "units": "cm^-2 s^-1 sr^-1 eV^-1",
        "datatype": datatype,
        "species": species,
    }

    if energy_range_ev is not None:
        elo, ehi = energy_range_ev
        attrs["energy_min_eV"] = float(elo)
        attrs["energy_max_eV"] = float(ehi)
        var_attrs["energy_min_eV"] = float(elo)
        var_attrs["energy_max_eV"] = float(ehi)

    ds = xr.Dataset(
        data_vars={
            "diff_number_flux": (
                ("time", "pitch_angle"),
                flux_t_pa,
                var_attrs,
            )
        },
        coords={
            "time": ("time", time.values),
            "pitch_angle": (
                "pitch_angle",
                pa_axis,
                {
                    "long_name": "Pitch angle",
                    "units": "deg",
                },
            ),
        },
        attrs=attrs,
    )

    return ds

In [ ]:
def energy_tag(elo, ehi):
    def fmt(x):
        if x >= 100:
            s = f"{x:.0f}"
        elif x >= 10:
            s = f"{x:.1f}"
        else:
            s = f"{x:.2f}"
        return s.replace(".", "p")

    return f"{fmt(elo)}_{fmt(ehi)}eV"

In [ ]:
def save_pa_flux_one_energy_bin(
    trange,
    edges,
    ibin,
    out_dir,
    *,
    datatype="peif",
    probe="a",
    mag_name="tha_fgl",
    timeout=300,
):
    """
    1つのenergy binについてPA differential number fluxを作り、NetCDFに保存する。

    edges: energy bin edges [eV], length = Nbin + 1
    ibin : 0 <= ibin < Nbin
    datatype: 'peif' or 'peef'
    """

    if datatype not in ["peif", "peef"]:
        raise ValueError("datatype must be 'peif' or 'peef'")

    edges = np.asarray(edges, dtype=float)

    elo = float(edges[ibin])
    ehi = float(edges[ibin + 1])

    tplot_name = f"th{probe}_{datatype}_flux_pa"

    commands = [
        f"trange = {trange}",
        (
            f"thm_part_products, probe='{probe}', datatype='{datatype}', "
            f"trange=trange, outputs='pa', units='flux', "
            f"energy=[{elo:.8g},{ehi:.8g}], mag_name='{mag_name}'"
        ),
        f"get_data, '{tplot_name}', data=d_pa",
        "pa_time = d_pa.x",
        "pa_flux = d_pa.y",
        "pa_angle = d_pa.v",
    ]

    tmpdir = run_idl_batch_simple(commands, timeout=timeout)

    pa_time = np.asarray(IDL.pa_time)
    pa_flux = np.asarray(IDL.pa_flux)
    pa_angle = np.asarray(IDL.pa_angle)

    ds = pa_flux_to_dataset(
        pa_time,
        pa_flux,
        pa_angle,
        datatype=datatype,
        probe=probe,
        energy_range_ev=(elo, ehi),
    )

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    tag = energy_tag(elo, ehi)
    nc_path = out_dir / f"th{probe}_{datatype}_flux_pa_{tag}.nc"

    ds.to_netcdf(nc_path)

    return nc_path, ds, tmpdir

In [ ]:
def save_pa_flux_all_energy_bins(
    trange,
    edges,
    out_dir,
    *,
    datatype="peif",
    probe="a",
    mag_name="tha_fgl",
    timeout_prepare=300,
    timeout_bin=300,
    prepare=True,
):
    """
    ESA ion/electronの各energy binについてPA flux productを作り、
    binごとにNetCDF保存する。
    """

    if datatype not in ["peif", "peef"]:
        raise ValueError("datatype must be 'peif' or 'peef'")

    edges = np.asarray(edges, dtype=float)

    if np.any(~np.isfinite(edges)):
        raise ValueError("edges contains NaN or inf")

    if np.any(edges <= 0):
        raise ValueError("energy edges must be positive")

    if not np.all(np.diff(edges) > 0):
        raise ValueError("energy edges must be strictly increasing")

    if prepare:
        print(f"Preparing THEMIS/SPEDAS data for {datatype}...", flush=True)
        prepare_themis_esa_for_pa(
            trange,
            datatype=datatype,
            probe=probe,
            timeout=timeout_prepare,
        )

    paths = []

    nbin = len(edges) - 1

    for ibin in range(nbin):
        elo = edges[ibin]
        ehi = edges[ibin + 1]

        print(
            f"[{datatype}] bin {ibin+1}/{nbin}: "
            f"{elo:.3g} - {ehi:.3g} eV",
            flush=True,
        )

        nc_path, ds, tmpdir = save_pa_flux_one_energy_bin(
            trange,
            edges,
            ibin,
            out_dir,
            datatype=datatype,
            probe=probe,
            mag_name=mag_name,
            timeout=timeout_bin,
        )

        paths.append(nc_path)

    return paths

In [ ]:
out_dir_peif = Path("/mnt/j/observation_data/themis/tha/idl_output/peif_pa/")
out_dir_peif.mkdir(parents=True, exist_ok=True)

peif_paths = save_pa_flux_all_energy_bins(
    trange,
    peif_edges,
    out_dir_peif,
    datatype="peif",
    probe="a",
    mag_name="tha_fgl",
    timeout_prepare=300,
    timeout_bin=300,
    prepare=True,
)

print(peif_paths[:3])
print(len(peif_paths))

In [ ]:
out_dir_peef = Path("/mnt/j/observation_data/themis/tha/idl_output/peef_pa/")
out_dir_peef.mkdir(parents=True, exist_ok=True)

peef_paths = save_pa_flux_all_energy_bins(
    trange,
    peef_edges,
    out_dir_peef,
    datatype="peef",
    probe="a",
    mag_name="tha_fgl",
    timeout_prepare=300,
    timeout_bin=300,
    prepare=True,
)

print(peef_paths[:3])
print(len(peef_paths))

In [ ]:
mpl.rcParams['font.size'] = 20

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import pytplot as pt
import matplotlib.ticker as mticker


def pa_plot_themis_nc(
    fig,
    ax,
    cax,
    nc_path,
    varname="diff_number_flux",
    vmin=None,
    vmax=None,
    percentile=(1, 99),
    ytitle="Pitch angle [deg]",
    ztitle=r"[cm$^{-2}$ s$^{-1}$ sr$^{-1}$ eV$^{-1}$]",
    cmap=cm.turbo,
    title=None,
):
    nc_path = Path(nc_path)
    ds = xr.open_dataset(nc_path)

    da = ds[varname].transpose("time", "pitch_angle")

    flux = np.asarray(da.values, dtype=float)                  # (T, PA)
    pa_axis = np.asarray(ds["pitch_angle"].values, dtype=float) # (PA,)
    t = pd.to_datetime(ds["time"].values)
    t_num = mdates.date2num(t.to_pydatetime())

    PA = np.broadcast_to(pa_axis[None, :], flux.shape)
    Tmesh = np.broadcast_to(t_num[:, None], flux.shape)

    good_PA = np.all(np.isfinite(PA), axis=0)
    if not np.all(good_PA):
        flux = flux[:, good_PA]
        PA = PA[:, good_PA]
        Tmesh = Tmesh[:, good_PA]

    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan

    valid = flux[np.isfinite(flux) & (flux > 0)]
    if valid.size == 0:
        raise ValueError(f"有効な正のfluxがない: {nc_path}")

    # ---- 各図ごとにpercentileでvmin/vmaxを決める ----
    if vmin is None:
        vmin = np.nanpercentile(valid, percentile[0])
    if vmax is None:
        vmax = np.nanpercentile(valid, percentile[1])

    if not np.isfinite(vmin) or not np.isfinite(vmax):
        raise ValueError(f"vmin/vmaxが非有限: vmin={vmin}, vmax={vmax}, file={nc_path}")

    if vmin <= 0:
        vmin = np.nanmin(valid)

    if not (vmin < vmax):
        vmax = vmin * 1.0001

    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)

    flux_plot = flux.copy()
    flux_plot[~np.isfinite(flux_plot)] = 1e-99

    mesh = ax.pcolormesh(
        Tmesh,
        PA,
        flux_plot,
        norm=norm,
        cmap=cmap,
        shading="nearest",
    )

    cb = fig.colorbar(mesh, cax=cax)
    cb.set_label(ztitle)

    ax.set_ylabel(ytitle)
    ax.grid(which="both", alpha=0.5)
    ax.minorticks_on()

    ax.set_ylim(0, 180)
    ax.set_yticks(np.arange(0, 181, 45))

    loc = mdates.AutoDateLocator()
    fmt = mdates.ConciseDateFormatter(loc)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)

    if title is None:
        emin = ds.attrs.get("energy_min_eV", None)
        emax = ds.attrs.get("energy_max_eV", None)
        datatype = ds.attrs.get("datatype", "")
        if emin is not None and emax is not None:
            title = f"{datatype}: {emin:.0f}–{emax:.0f} eV"
        else:
            title = nc_path.name

    ax.set_title(title)

    print(f"{nc_path.name}: vmin={vmin:.3e}, vmax={vmax:.3e}", flush=True)

    return fig, ax, cax

In [ ]:
def get_global_vmin_vmax_from_nc_files(
    nc_files,
    varname="diff_number_flux",
    percentile=None,
):
    vals = []

    for f in nc_files:
        ds = xr.open_dataset(f)
        da = ds[varname]
        arr = np.asarray(da.values, dtype=float)
        arr = arr[np.isfinite(arr) & (arr > 0)]
        if arr.size > 0:
            vals.append(arr)

    if len(vals) == 0:
        raise ValueError("有効な正のfluxが全ファイルで見つからない。")

    vals = np.concatenate(vals)

    if percentile is None:
        vmin = np.nanmin(vals)
        vmax = np.nanmax(vals)
    else:
        pmin, pmax = percentile
        vmin = np.nanpercentile(vals, pmin)
        vmax = np.nanpercentile(vals, pmax)

    if not (vmin < vmax):
        vmax = vmin * 1.0001

    return vmin, vmax

In [ ]:
def plot_all_pa_flux_nc_in_dir(
    in_dir,
    out_dir=None,
    varname="diff_number_flux",
    vmin=None,
    vmax=None,
    percentile=(1, 99),
    figsize=(10, 4),
    dpi=200,
):
    in_dir = Path(in_dir)
    nc_files = sorted(in_dir.glob("*.nc"))

    if len(nc_files) == 0:
        raise FileNotFoundError(f"No .nc files found in {in_dir}")

    if out_dir is None:
        out_dir = in_dir / "plots"
    else:
        out_dir = Path(out_dir)

    out_dir.mkdir(parents=True, exist_ok=True)

    saved = []

    for nc_path in nc_files:
        print(f"Plotting: {nc_path.name}", flush=True)

        fig = plt.figure(figsize=figsize)
        gs = fig.add_gridspec(
            nrows=1,
            ncols=2,
            width_ratios=[1, 0.025],
            wspace=0.05,
        )

        ax = fig.add_subplot(gs[0, 0])
        cax = fig.add_subplot(gs[0, 1])

        pa_plot_themis_nc(
            fig,
            ax,
            cax,
            nc_path,
            varname=varname,
            vmin=vmin,
            vmax=vmax,
            percentile=percentile,
        )

        png_path = out_dir / (nc_path.stem + ".png")
        fig.savefig(png_path, dpi=dpi, bbox_inches="tight")
        plt.close(fig)

        saved.append(png_path)

    return saved

In [ ]:
out_dir_peef = Path("/mnt/j/observation_data/themis/tha/idl_output/peef_pa/")
out_dir_peef_fig = Path("/mnt/j/KAW_observation/ESA_pitch_angle_THEMIS_A/20220901/electron/")
out_dir_peef_fig.mkdir(parents=True, exist_ok=True)

png_paths = plot_all_pa_flux_nc_in_dir(
    in_dir=out_dir_peef,
    out_dir=out_dir_peef_fig,
    varname="diff_number_flux",
    percentile=(50, 99),
    figsize=(15, 3),
    dpi=200,
)

print(f"Saved {len(png_paths)} figures")
print(png_paths[:3])

In [ ]:
out_dir_peif = Path("/mnt/j/observation_data/themis/tha/idl_output/peif_pa/")
out_dir_peif_fig = Path("/mnt/j/KAW_observation/ESA_pitch_angle_THEMIS_A/20220901/ion/")
out_dir_peif_fig.mkdir(parents=True, exist_ok=True)

png_paths = plot_all_pa_flux_nc_in_dir(
    in_dir=out_dir_peif,
    out_dir=out_dir_peif_fig,
    varname="diff_number_flux",
    percentile=(50, 99),
    figsize=(15, 3),
    dpi=200,
)

print(f"Saved {len(png_paths)} figures")
print(png_paths[:3])